In [0]:
from pyspark.sql import functions as F

inv      = spark.table("workspace.bronze.inventory")
stores   = spark.table("workspace.bronze.stores")
products = spark.table("workspace.bronze.products")

# 1. Type casting. try_cast gives NULL on bad values; plain cast would fail the job.
typed = inv.select(
    F.expr("try_cast(store_id as int)").alias("store_id"),
    F.expr("try_cast(product_id as int)").alias("product_id"),
    F.expr("try_cast(stock_qty as int)").alias("stock_qty"),
    F.expr("try_cast(reorder_level as int)").alias("reorder_level"),
    F.expr("try_to_date(snapshot_date)").alias("snapshot_date"),
    "_source_file", "_ingested_at",
)

# 2. Reference keys for the orphan check
s = stores.select(F.col("store_id").cast("int").alias("s_id")).distinct().withColumn("_s", F.lit(1))
p = products.select(F.col("id").cast("int").alias("p_id")).distinct().withColumn("_p", F.lit(1))

flagged = (typed
    .join(s, typed.store_id == s.s_id, "left")
    .join(p, typed.product_id == p.p_id, "left")
    .withColumn("reject_reason",
        F.when(F.col("stock_qty").isNull(), "null_stock_qty")
         .when(F.col("stock_qty") < 0, "negative_stock_qty")
         .when(F.col("_s").isNull(), "unknown_store")
         .when(F.col("_p").isNull(), "unknown_product")))

helper = ["s_id", "_s", "p_id", "_p"]
rejects = flagged.filter("reject_reason IS NOT NULL").drop(*helper)
clean   = flagged.filter("reject_reason IS NULL").drop(*helper, "reject_reason")

# 3. Dedupe on the business key, after validation
silver = clean.dropDuplicates(["store_id", "product_id", "snapshot_date"])

# 4. Write
(silver.write.format("delta").mode("overwrite").option("overwriteSchema","true")
       .saveAsTable("workspace.silver.inventory"))
(rejects.write.format("delta").mode("overwrite").option("overwriteSchema","true")
       .saveAsTable("workspace.silver.inventory_rejects"))

# 5. Reconcile: every Bronze row must be accounted for
b, sv, rj = inv.count(), silver.count(), rejects.count()
print(f"bronze={b}  silver={sv}  rejects={rj}  duplicates_dropped={b - sv - rj}")

bronze=1428  silver=1400  rejects=21  duplicates_dropped=7


In [0]:
from pyspark.sql import functions as F
for t in ["stores", "customers", "orders", "order_items", "products"]:
    df = spark.table(f"workspace.bronze.{t}")
    print(f"\n=== {t}: {df.count()} rows ===")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns if not c.startswith("_")]).show()


=== stores: 10 rows ===
+--------+----------+----+-----+-----------+
|store_id|store_name|city|state|opened_date|
+--------+----------+----+-----+-----------+
|       0|         0|   0|    0|          0|
+--------+----------+----+-----+-----------+


=== customers: 200 rows ===
+-----------+----------+---------+-----+-----+----+----------+----------+
|customer_id|first_name|last_name|email|phone|city|created_at|updated_at|
+-----------+----------+---------+-----+-----+----+----------+----------+
|          0|         0|        0|    0|    0|   0|         0|         0|
+-----------+----------+---------+-----+-----+----+----------+----------+


=== orders: 4000 rows ===
+--------+-----------+--------+----------+------------+------------+----------+
|order_id|customer_id|store_id|order_date|payment_mode|order_status|updated_at|
+--------+-----------+--------+----------+------------+------------+----------+
|       0|          0|       0|         0|           0|           0|         0|
+-